In [ ]:
## LOAD DATA

import os
import re
import pandas as pd

headers_list = [
    'Year', 'Zone Substation', 'Date', 'Unit',
    '00:15', '00:30', '00:45', '01:00', '01:15', '01:30', '01:45',
    '02:00', '02:15', '02:30', '02:45', '03:00', '03:15', '03:30', '03:45',
    '04:00', '04:15', '04:30', '04:45', '05:00', '05:15', '05:30', '05:45',
    '06:00', '06:15', '06:30', '06:45', '07:00', '07:15', '07:30', '07:45',
    '08:00', '08:15', '08:30', '08:45', '09:00', '09:15', '09:30', '09:45',
    '10:00', '10:15', '10:30', '10:45', '11:00', '11:15', '11:30', '11:45',
    '12:00', '12:15', '12:30', '12:45', '13:00', '13:15', '13:30', '13:45',
    '14:00', '14:15', '14:30', '14:45', '15:00', '15:15', '15:30', '15:45',
    '16:00', '16:15', '16:30', '16:45', '17:00', '17:15', '17:30', '17:45',
    '18:00', '18:15', '18:30', '18:45', '19:00', '19:15', '19:30', '19:45',
    '20:00', '20:15', '20:30', '20:45', '21:00', '21:15', '21:30', '21:45',
    '22:00', '22:15', '22:30', '22:45', '23:00', '23:15', '23:30', '23:45',
    '24:00'
]

def load_year_data(year: int, base_dir: str = '.') -> dict:
    """
    Load all CSVs in base_dir/<year>/ ending with FY<year>.csv,
    rename columns to headers_list, extract only the leading alphabetic
    suburb name, and—if a suburb has multiple files—sum each numeric
    column across them. Returns a dict of DataFrames named
    df_<Suburb>_<Year>_RAW.
    """
    year_dir = os.path.join(base_dir, str(year))
    pattern = re.compile(rf'^(?P<suburb>[^\d_]+).*FY{year}\.csv$', re.IGNORECASE)

    suburb_acc = {}
    for fname in os.listdir(year_dir):
        m = pattern.match(fname)
        if not m:
            continue

        raw = m.group('suburb').strip()
        suburb = re.sub(r'\W+', '', raw).title()
        path = os.path.join(year_dir, fname)

        df = pd.read_csv(path)

        df.columns = headers_list

        if suburb in suburb_acc:

            base = suburb_acc[suburb]
            num_cols = df.select_dtypes('number').columns
            base[num_cols] = base[num_cols].add(df[num_cols], fill_value=0)
            suburb_acc[suburb] = base
        else:
            suburb_acc[suburb] = df.copy()

    dfs = {}
    for suburb, df in suburb_acc.items():
        var_name = f"df_{suburb}_{year}_RAW"
        dfs[var_name] = df
        globals()[var_name] = df

    return dfs

def load_2019_data(base_dir: str = '.') -> dict:
    return load_year_data(2019, base_dir)

def load_2020_data(base_dir: str = '.') -> dict:
    return load_year_data(2020, base_dir)

def load_2021_data(base_dir: str = '.') -> dict:
    return load_year_data(2021, base_dir)

def load_2022_data(base_dir: str = '.') -> dict:
    return load_year_data(2022, base_dir)

def load_2023_data(base_dir: str = '.') -> dict:
    return load_year_data(2023, base_dir)

def load_2024_data(base_dir: str = '.') -> dict:
    return load_year_data(2024, base_dir)

dfs_2019 = load_2019_data(base_dir='POWER DATA')
dfs_2020 = load_2020_data(base_dir='POWER DATA')
dfs_2021 = load_2021_data(base_dir='POWER DATA')
dfs_2022 = load_2022_data(base_dir='POWER DATA')
dfs_2023 = load_2023_data(base_dir='POWER DATA')
dfs_2024 = load_2024_data(base_dir='POWER DATA')

In [ ]:
## CHECKING MISSING YEARS SUBURBS DATA

import re
from collections import defaultdict

def check_loaded_suburb_years(start=2019, end=2024):
    """
    Check which suburbs missing what years of data and list suburbs with all the data
    """
    pattern = re.compile(r'^df_([A-Za-z]+)_(\d{4})_RAW$')
    expected = set(range(start, end+1))
    presence = defaultdict(set)
    
    for varname in globals():
        m = pattern.match(varname)
        if not m:
            continue
        suburb, year = m.group(1), int(m.group(2))
        if start <= year <= end:
            presence[suburb].add(year)
    
    missing = {
        suburb: sorted(expected - years)
        for suburb, years in presence.items()
        if expected - years
    }
    
    full_data_suburbs = [
        suburb for suburb, years in presence.items() if years == expected
    ]
    
    return missing, full_data_suburbs


missing_years, full_data_suburbs = check_loaded_suburb_years()

if not missing_years:
    print("Every suburb has data for all years 2019–2024.\n")
else:
    print("Suburbs with missing years:\n")
    for suburb, yrs in missing_years.items():
        print(f" • {suburb:15s} missing: {yrs}")

print("\nSuburbs with data for all years 2019–2024:")
if full_data_suburbs:
    for suburb in full_data_suburbs:
        print(f" • {suburb:15s} has data for all years.")
else:
    print("No suburb has data for all years.")


In [ ]:
## LIST SUBSTATIONS
import re

def list_all_suburbs():
    """
    Lists all suburbs based on variables named df_<Suburb>_<Year>_RAW in globals().
    """
    pattern = re.compile(r'^df_([A-Za-z]+)_(\d{4})_RAW$')
    suburbs = set()
    
    for varname in globals():
        m = pattern.match(varname)
        if m:
            suburb = m.group(1)
            suburbs.add(suburb)
    
    return sorted(suburbs)

all_suburbs = list_all_suburbs()
print("All suburbs:")
for suburb in all_suburbs:
    print(f" • {suburb}")

In [ ]:
## GROUP DATA FROM 2019 TO 2024

import pandas as pd
import re
from collections import defaultdict

def combine_loaded_by_suburb(dfs_by_year):
    """
    Takes a dict of per-year loaded DataFrames (e.g. {2019: dfs_2019, …})
    and concatenates all years for each suburb into one DataFrame.
    Parses the 'Date' column handling both 'YYYY-MM-DD' and 'DDMMMYYYY' formats.
    Returns a dict mapping df_<Suburb>_RAW to the combined DataFrame,
    and injects each into globals().
    """
    suburb_buffers = defaultdict(list)

    for year, dfs in dfs_by_year.items():
        for var_name, df in dfs.items():
            suburb = var_name.split('_')[1]
            temp = df.copy()
            d1 = pd.to_datetime(temp['Date'], format='%Y-%m-%d', errors='coerce')
            d2 = pd.to_datetime(temp['Date'], format='%d%b%Y', errors='coerce', dayfirst=True)
            temp['Date'] = d1.fillna(d2)
            if temp['Date'].isna().any():
                unmatched = temp.loc[temp['Date'].isna(), 'Date'].unique()
                print(f"Warning: Could not parse these dates for {var_name}: {unmatched}")
            temp['Year'] = year
            suburb_buffers[suburb].append(temp)

    combined = {}
    for suburb, df_list in suburb_buffers.items():
        df_all = pd.concat(df_list, ignore_index=True)
        df_all = df_all.sort_values('Date')
        var_name = f"df_{suburb}_RAW"
        globals()[var_name] = df_all
        combined[var_name] = df_all

    return combined

dfs_by_year = {
  2019: dfs_2019,
  2020: dfs_2020,
  2021: dfs_2021,
  2022: dfs_2022,
  2023: dfs_2023,
  2024: dfs_2024,
}
dfs_all = combine_loaded_by_suburb(dfs_by_year)



In [ ]:
## ROUND NUMERIC COLUMNS

def round_numeric_columns(dfs_suburb):
    """
    Rounds all numeric columns to 6 decimal places in each DataFrame.
    dfs_suburb: dict mapping 'df_<Suburb>_RAW' -> DataFrame
    """
    for var_name, df in dfs_suburb.items():
        num_cols = df.select_dtypes(include='number').columns
        df[num_cols] = df[num_cols].round(5)

round_numeric_columns(dfs_all)

In [ ]:
## CREATE A MISSING VALUE SUMMARY FOR EACH SUBURBS

import pandas as pd
import re

pattern = re.compile(r'^df_([A-Za-z]+)_RAW$')
dfs_all = {
    var: df for var, df in globals().items()
    if pattern.match(var)
}

counts = {}
for var_name, df in dfs_all.items():
    suburb = var_name.split('_')[1]
    num_cols = df.select_dtypes(include='number').columns
    counts[suburb] = df[num_cols].isna().sum()

missing_summary_df = pd.DataFrame.from_dict(
    counts, orient='index'
).fillna(0).astype(int)

missing_summary_df.index.name = 'Suburb'
missing_summary_df.columns.name = 'Numeric Columns'
missing_summary_df = missing_summary_df.sort_index()

display(missing_summary_df)

In [ ]:
## CLEAN DATA USING VARIOUS METHODS

import pandas as pd
import re

def impute_missing_for_all_suburbs():
    """
    Scans globals() snapshot for df_<Suburb>_RAW DataFrames,
    applies multiple imputation methods without mutating globals during iteration,
    and injects results as df_<Suburb>_<METHOD> into globals().
    Returns a nested dict imputed[suburb][method] = DataFrame.
    """
    methods = {
        'ffill': lambda df: df.fillna(method='ffill'),
        'bfill': lambda df: df.fillna(method='bfill'),
        'mean': lambda df: df.fillna(df.mean()),
        'median': lambda df: df.fillna(df.median()),
        'linear': lambda df: df.interpolate(method='linear'),
    }
    pattern = re.compile(r'^df_([A-Za-z]+)_RAW$')
    imputed = {}

    global_items = list(globals().items())

    for var_name, df in global_items:
        if not pattern.match(var_name) or not isinstance(df, pd.DataFrame):
            continue
        suburb = pattern.match(var_name).group(1)
        imputed[suburb] = {}
        for name, func in methods.items():
            df_copy = df.copy()
            num_cols = df_copy.select_dtypes(include='number').columns
            df_copy[num_cols] = func(df_copy[num_cols])
            imputed[suburb][name] = df_copy
            globals()[f"df_{suburb}_{name.upper()}"] = df_copy

    return imputed

imputed_results = impute_missing_for_all_suburbs()



In [ ]:
## RENAME FINAL DATA
import re

for var in list(globals()):
    m = re.match(r'^df_([A-Za-z]+)_LINEAR$', var)
    if m:
        suburb = m.group(1)
        new_name = f"df_{suburb}_FINAL"
        globals()[new_name] = globals()[var]
        del globals()[var]


In [ ]:
## AGGREGATE POWER DATA AND SUM UP NUMERIC COLUMNS

import re
import pandas as pd

def finalize_suburb_data():
    """
    For each DataFrame named df_<Suburb>_FINAL in globals():
      - Compute "Total Usage" = sum of all numeric columns
      - Compute "Daily Peak Usage" = max of those same columns
      - Drop the original timeslot numeric columns
      - Rename the DataFrame in globals() to df_<Suburb>_Z
    """
    pattern = re.compile(r'^df_([A-Za-z]+)_FINAL$')
    for var_name, df in list(globals().items()):
        m = pattern.match(var_name)
        if not m or not isinstance(df, pd.DataFrame):
            continue
        
        suburb = m.group(1)
        df_copy = df.copy()
        
        numeric_cols = [
            col for col in df_copy.select_dtypes(include='number').columns
            if col != 'Year']
        
        df_copy['Total Daily Usage'] = df_copy[numeric_cols].sum(axis=1)
        df_copy['Daily Peak Usage'] = df_copy[numeric_cols].max(axis=1)
        
        df_copy.drop(columns=numeric_cols, inplace=True)
        
        new_var = f"df_{suburb}_Z"
        globals()[new_var] = df_copy
        del globals()[var_name]

finalize_suburb_data()




In [ ]:
## FIND TARGETED POWER DATA AND DELETE THE REST

import re

def prune_power_dataframes(target_suburbs):
    """
    1) Reports which target_suburbs have a df_<Suburb>_Z in globals()
    2) Deletes every df_<Suburb>_<SUFFIX> var for any Suburb not found
    3) For each found Suburb, deletes all df_<Suburb>_<SUFFIX> except _Z
    """
    cleaned_targets = {
        re.sub(r'\W+', '', s).title(): s
        for s in target_suburbs
    }
    
    z_pat    = re.compile(r'^df_([A-Za-z]+)_Z$')
    existing = {
        m.group(1)
        for var in globals()
        if (m := z_pat.match(var))
    }
    found   = sorted(c for c in existing if c in cleaned_targets)
    missing = sorted(c for c in cleaned_targets if c not in existing)
    
    if found:
        print("✅ Found DataFrames for suburbs:")
        for c in found:
            print(f"  • {cleaned_targets[c]} (df_{c}_Z)")
    else:
        print("⚠️  No target DataFrames were found.")
    if missing:
        print("\n🔴 Missing DataFrames for suburbs:")
        for c in missing:
            print(f"  • {cleaned_targets[c]}")
    else:
        print("\n✅ All target suburbs were found.")
    
    suffixes = [
        '2019_RAW','2020_RAW','2021_RAW','2022_RAW','2023_RAW','2024_RAW',
        'BFILL','FFILL','MEAN','MEDIAN','RAW','Z'
    ]

    suf_group = '|'.join(suffixes)
    del_pat   = re.compile(rf'^df_([A-Za-z]+)_({suf_group})$')

    for var in list(globals()):
        if (m := del_pat.match(var)):
            suburb = m.group(1)
            if suburb not in found:
                del globals()[var]
    
    for suburb in found:
        for suf in suffixes:
            if suf == 'Z':
                continue
            var = f"df_{suburb}_{suf}"
            if var in globals():
                del globals()[var]


target_suburbs = [
    'Epping', 'Macquarie Park', 'Bankstown',
    'Concord', 'Strathfield South', 'Burwood', 'Lidcombe',
    'Campsie', 'Marrickville', 'Leichhardt',
    'Crows Nest', 'Surry Hills', 'City Central', 'Camperdown',
    'Mascot', 'Rockdale', 'North Sydney',
    'Auburn', 'Hurstville North',
    'Revesby',
    'Terrey Hills',
    'Mayfield West', 'Newcastle CBD', 'Maryland',
]

prune_power_dataframes(target_suburbs)

list_suburbs = sorted([var for var in globals() if var.startswith('df_') and var.endswith('_Z')])
print(list_suburbs)

print(len(list_suburbs))

# ayushmaan.tomar@sydney.edu.au


In [ ]:
## LOAD CLIMATE DATA

import os
import re
import pandas as pd

def load_weather_data(base_dir: str = 'WEATHER DATA') -> dict:
    """
    Walks through base_dir, finds sub-folders like
      '66059 - Terrey Hills AWS',
    and in each reads every .csv it finds (min/max temp, rainfall, solar),
    loads each into its own DataFrame, and returns:

    It also injects globals named
      df_<StationName>_<METRIC>
    for quick interactive use.
    """
    weather_data = {}
    for folder in os.listdir(base_dir):
        station_path = os.path.join(base_dir, folder)
        if not os.path.isdir(station_path):
            continue

        if ' - ' in folder:
            _, name = folder.split(' - ', 1)
        else:
            name = folder

        station = re.sub(r'\W+', '', name).title()
        weather_data[station] = {}

        for fname in os.listdir(station_path):
            if not fname.lower().endswith('.csv'):
                continue
            path = os.path.join(station_path, fname)
            df = pd.read_csv(path)

            metric = fname.split('_', 1)[0].lower()

            weather_data[station][metric] = df

            globals()[f"df_{station}_{metric.upper()}"] = df

    return weather_data

weather_data = load_weather_data('WEATHER DATA')


In [ ]:
## COMBINE TIME FOR CLIMATE DATA AND PARSE DATETIME

import pandas as pd

def parse_weather_dates(weather_dict):
    """
    Given weather_dict as returned by load_weather_data(),
    for each station and each metric DataFrame:
      • Combines Year/Month/Day into a datetime Date column
      • Drops the original Year, Month, Day columns
    Returns the same dict (modified in place).
    """
    for station, metrics in weather_dict.items():
        for metric, df in metrics.items():
            df['Date'] = pd.to_datetime({
                'year':  df['Year'],
                'month': df['Month'],
                'day':   df['Day']
            })
            df.drop(columns=['Year', 'Month', 'Day'], inplace=True)
    return weather_dict

weather_data = parse_weather_dates(weather_data)

In [ ]:
## DROP UNECESSARY AND UNUSED VARIABLES FROM CLIMATE DATA

import re
import pandas as pd

def clean_loaded_weather_metrics():
    drop_map = {
        'MINTEMP': ['Days of accumulation of minimum temperature', 'Quality'],
        'MAXTEMP': ['Days of accumulation of maximum temperature', 'Quality'],
        'RAINFALL': ['Period over which rainfall was measured (days)', 'Quality']
    }

    for var_name, df in list(globals().items()):
        if not isinstance(df, pd.DataFrame):
            continue
        m = re.match(r'^df_[A-Za-z]+_(MINTEMP|MAXTEMP|RAINFALL)$', var_name)
        if not m:
            continue

        metric = m.group(1)
        to_drop = [col for col in drop_map[metric] if col in df.columns]
        if to_drop:
            df.drop(columns=to_drop, inplace=True)
            globals()[var_name] = df


clean_loaded_weather_metrics()

In [ ]:
## PORTION THE CLIMATE DATA BASED ON TIME PERIOD

import re
import pandas as pd

def cut_weather_dfs_by_date(start='2018-05-01 00:00:00', end='2024-04-30 00:00:00'):
    """
    Scans globals() for DataFrames named df_<Station>_MINTEMP,
    df_<Station>_MAXTEMP, df_<Station>_RAINFALL, or df_<Station>_SOLAR,
    and keeps only rows where 'Date' lies between start and end.
    """
    start_ts = pd.to_datetime(start)
    end_ts   = pd.to_datetime(end)
    pattern  = re.compile(r'^df_.*_(MINTEMP|MAXTEMP|RAINFALL|SOLAR)$')

    for var_name, df in list(globals().items()):
        if not pattern.match(var_name):
            continue
        if not isinstance(df, pd.DataFrame) or 'Date' not in df.columns:
            continue

        mask = (df['Date'] >= start_ts) & (df['Date'] <= end_ts)
        trimmed = df.loc[mask].copy()
        globals()[var_name] = trimmed

cut_weather_dfs_by_date()

In [ ]:
## MERGE WEATHER DATA AND IMPUTE MISSING VALUES

import re
import pandas as pd

def merge_station_weather():
    """
    For each station, merges the four metric DataFrames on Date into
    df_<Station>_WEATHER, then deletes the single-metric globals.
    Missing data are imputed using linear interpolation.
    """
    # Pattern to identify station metric DataFrame globals
    pattern = re.compile(r'^df_([A-Za-z]+)_(MINTEMP|MAXTEMP|RAINFALL|SOLAR)$')
    col_map = {
        'MINTEMP': 'Minimum temperature (Degree C)',
        'MAXTEMP': 'Maximum temperature (Degree C)',
        'RAINFALL': 'Rainfall amount (millimetres)',
        'SOLAR': 'Daily global solar exposure (MJ/m*m)'
    }

    # Group available metric DataFrames by station
    station_metrics = {}
    for var_name, df in list(globals().items()):
        match = pattern.match(var_name)
        if match and isinstance(df, pd.DataFrame):
            station, metric = match.group(1), match.group(2)
            station_metrics.setdefault(station, {})[metric] = var_name

    # Merge and interpolate for each station
    for station, metrics in station_metrics.items():
        merged_df = None
        for metric, var_name in metrics.items():
            df = globals()[var_name]
            col_name = col_map[metric]
            temp = df[['Date', col_name]].copy()
            temp.rename(columns={col_name: metric.lower()}, inplace=True)

            if merged_df is None:
                merged_df = temp
            else:
                merged_df = pd.merge(
                    merged_df,
                    temp,
                    on='Date',
                    how='outer'
                )

        # Ensure Date is datetime and sort
        merged_df['Date'] = pd.to_datetime(merged_df['Date'])
        merged_df.sort_values('Date', inplace=True)

        # Set Date as index for interpolation
        merged_df.set_index('Date', inplace=True)

        # Linear interpolation across all numeric columns, filling start/end gaps
        merged_df.interpolate(
            method='linear',
            limit_direction='both',
            inplace=True
        )

        # Reset index to bring Date back as a column
        merged_df.reset_index(inplace=True)

        # Assign merged and imputed DataFrame to global
        globals()[f"df_{station}_WEATHER"] = merged_df

        # Clean up individual metric DataFrames
        for var_name in metrics.values():
            del globals()[var_name]

merge_station_weather()

In [ ]:
## MERGE CLIMATE AND POWER DATA

import pandas as pd
import re

weather_station_substations = {
    "Parramatta North Masons Dr": ["Epping", "Macquarie Park", "Bankstown"],
    "Sydney Olympic Park AWS Archery Centre": ["Concord", "Strathfield South", "Burwood", "Lidcombe"],
    "Canterbury Racecourse AWS NSW": ["Campsie", "Marrickville", "Leichhardt"],
    "Sydney Observatory Hill": ["Crows Nest", "Surry Hills", "City Central", "Camperdown"],
    "Sydney Airport": ["Mascot", "Rockdale", "North Sydney"],
    "Bankstown Airport": ["Auburn", "Hurstville North", "Bankstown"],
    "Holsworthy Aerodrome AWS": ["Revesby"],
    "Terrey Hills AWS": ["Terrey Hills"],
    "Newcastlenobbyssignalstationaws": ["Mayfield West", "Newcastle CBD", "Maryland"]
}

def clean(name: str) -> str:
    return re.sub(r'\s+', '', name).lower()

weather_dfs    = {}
substation_dfs = {}

for varname, df in globals().items():
    m = re.match(r'^df_(.+)_WEATHER$', varname, re.IGNORECASE)
    if m:
        weather_dfs[clean(m.group(1))] = df

for varname, df in globals().items():
    m = re.match(r'^df_(.+)_Z$', varname, re.IGNORECASE)
    if m:
        substation_dfs[clean(m.group(1))] = df

for ws_name, subs in weather_station_substations.items():
    wkey = clean(ws_name)
    wdf = weather_dfs[wkey][["Date", "mintemp", "maxtemp", "rainfall", "solar"]]

    for sub_name in subs:
        skey = clean(sub_name)
        sdf  = substation_dfs[skey]

        merged = pd.merge(
            sdf,
            wdf,
            on="Date",
            how="left"
        )

        no_space = ''.join(sub_name.split())
        final_var = f"df_{no_space}_FINAL"
        globals()[final_var] = merged


In [ ]:
## DELETE FINAL EXCESS VARIABLES

import re

to_delete = [
    var for var in globals() 
    if re.match(r'^df_.*_Z$', var, re.IGNORECASE) 
    or re.match(r'^df_.*_WEATHER$', var, re.IGNORECASE)
]
for var in to_delete:
    del globals()[var]

final_vars = sorted([
    var for var in globals() 
    if re.match(r'^df_.*_FINAL$', var, re.IGNORECASE)
])
print(final_vars)
print(len(final_vars))

globals()['df_Sydney_FINAL'] = globals().pop('df_CityCentral_FINAL')
globals()['df_Newcastle_FINAL'] = globals().pop('df_NewcastleCBD_FINAL')
globals()['df_Hurstville_FINAL'] = globals().pop('df_HurstvilleNorth_FINAL')

In [ ]:
## EXTRA FEATURES ADDED
import pandas as pd

# Identify all suburb DataFrames in globals (ending with _FINAL)
suburb_dfs = {name: df for name, df in globals().items()
              if name.startswith('df_') and name.endswith('_FINAL') and isinstance(df, pd.DataFrame)}

# Concatenate to compute global temperature thresholds
df_all = pd.concat(suburb_dfs.values(), ignore_index=True)
hot_thr = df_all['maxtemp'].quantile(0.90)
cold_thr = df_all['mintemp'].quantile(0.10)

# Define a run-length helper
def run_length(series):
    # series: binary 0/1 indicator of event
    return series * (series.groupby((series != series.shift()).cumsum())
                           .cumcount().add(1))

# Loop through each suburb DataFrame and add features
for name, df in suburb_dfs.items():
    # Average temperature
    df['T_avg'] = (df['maxtemp'] + df['mintemp']) / 2

    # Degree days based on 18°C threshold
    df['HDD_18'] = (18 - df['T_avg']).clip(lower=0)
    df['CDD_18'] = (df['T_avg'] - 18).clip(lower=0)

    # Rolling degree-day sums over 3, 7, and 14 days
    for window in (3, 7, 14):
        df[f'HDD_{window}d'] = df['HDD_18'].rolling(window).sum()
        df[f'CDD_{window}d'] = df['CDD_18'].rolling(window).sum()

    # Heatwave/cold-spell indicators and streak lengths
    df['heatwave_day'] = (df['maxtemp'] > hot_thr).astype(int)
    df['coldspell_day'] = (df['mintemp'] < cold_thr).astype(int)
    df['heatwave_run'] = run_length(df['heatwave_day'])
    df['coldspell_run'] = run_length(df['coldspell_day'])

    # Assign back to globals
    globals()[name] = df

# Optionally, update df_all with new features as well
globals()['df_all'] = df_all.copy()


dfs = []
target_suburbs = [
    'Epping', 'Macquarie Park', 'Bankstown', 'Concord',
    'Strathfield South', 'Burwood', 'Lidcombe', 'Campsie',
    'Marrickville', 'Leichhardt', 'Crows Nest', 'Surry Hills',
    'Sydney', 'Camperdown', 'Mascot', 'Rockdale',
    'North Sydney', 'Auburn', 'Hurstville', 'Revesby',
    'Terrey Hills', 'Mayfield West', 'Newcastle'
]

for suburb in target_suburbs:
    var = f"df_{suburb.replace(' ', '')}_FINAL"
    df_sub = globals()[var].copy()
    df_sub['suburb'] = suburb
    dfs.append(df_sub)
df_all = pd.concat(dfs, ignore_index=True)


In [ ]:
from bokeh.io import output_notebook, output_file, show
from bokeh.plotting import figure
from bokeh.layouts import row
from bokeh.models import ColumnDataSource
import numpy as np

output_notebook()
output_file("histogram usage.html", title="histogram usage")

df_all['mean_temp'] = (df_all['mintemp'] + df_all['maxtemp']) / 2

agg = df_all.groupby('Date').agg({
    'Total Daily Usage': 'mean',
    'mean_temp':   'mean',
    'rainfall':    'mean',
    'solar':       'mean'
}).reset_index()

PW, PH = 400, 300

def scatter_with_fit(x, y, x_label, y_label="Daily Usage (MW)"):
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ys = m*xs + b

    p = figure(width=PW, height=PH,
               title=f"{y_label} vs {x_label}",
               tools="pan,wheel_zoom,box_zoom,reset,save,hover")
    src = ColumnDataSource(dict(x=x, y=y))
    p.circle('x','y', source=src, size=5, alpha=0.3)
    p.line(xs, ys, line_width=2, color="orange")
    p.xaxis.axis_label = x_label
    p.yaxis.axis_label = y_label
    p.hover.tooltips = [
        (x_label, "@x{0.2f}"),
        (y_label, "@y{0.2f}")
    ]
    return p

# build and show the row of three plots
agg_plots = [
    scatter_with_fit(agg['mean_temp'],       agg['Total Daily Usage'], "Mean Temp (°C)"),
    scatter_with_fit(agg['rainfall'],        agg['Total Daily Usage'], "Rainfall (mm)"),
    scatter_with_fit(agg['solar'],           agg['Total Daily Usage'], "Solar (kW/m²)"),
]
show(row(*agg_plots))


In [ ]:
## CORRELATION MATRIX PLOT
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, LinearColorMapper, ColorBar, BasicTicker
from bokeh.palettes import Viridis256
import numpy as np

# render inline
output_notebook()

agg['Date'] = pd.to_datetime(agg['Date'])

# 1) pick your variables
corr_vars = ['mean_temp','rainfall','solar','Total Daily Usage']

# 2) compute correlation matrix and melt to long form
corr_matrix = df_all[corr_vars].corr()
corr_df = corr_matrix.stack().reset_index()
corr_df.columns = ['var1','var2','corr']

# 3) set up color mapper
mapper = LinearColorMapper(palette=Viridis256, low=-1, high=1)

# 4) figure
p_corr = figure(
    width=600, height=500,
    title="Correlation Matrix",
    x_range=corr_vars, y_range=list(reversed(corr_vars)),
    tools="hover,save,reset"
)

# 5) draw rectangles
p_corr.rect(
    x='var2', y='var1', width=1, height=1,
    source=ColumnDataSource(corr_df),
    fill_color={'field':'corr','transform':mapper},
    line_color=None
)

# 6) add color bar
color_bar = ColorBar(
    color_mapper=mapper,
    ticker=BasicTicker(desired_num_ticks=10),
    label_standoff=6,
    border_line_color=None,
    location=(0,0)
)
p_corr.add_layout(color_bar, 'right')

# 7) styling & hover
p_corr.xaxis.major_label_orientation = np.pi/4
p_corr.hover.tooltips = [
    ("Pair", "@var1 ↔ @var2"),
    ("ρ",    "@corr{0.2f}")
]

# 8) show it
show(p_corr)


In [ ]:
## DAILY USAGE ROLLING MEAN PLOT

from bokeh.io import output_notebook,show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
from bokeh.models import HoverTool
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

output_notebook()


# —————————————————————————————
# Prepare time series data (Date column guaranteed)
# —————————————————————————————
agg_ts = agg.copy()
agg_ts['Date'] = pd.to_datetime(agg_ts['Date'])
dates = agg_ts['Date']

usage  = agg_ts['Total Daily Usage']
rolling = usage.rolling(30).mean()

# —————————————————————————————
# Plot 1: Daily usage + 30-day rolling mean
# —————————————————————————————
src1 = ColumnDataSource(dict(date=dates, usage=usage, rolling=rolling))

p1 = figure(
    x_axis_type='datetime', width=1000, height=400,
    title='Daily Usage & 30-Day Rolling Mean',
    tools='pan,wheel_zoom,box_zoom,reset,save,hover'
)

hover = p1.select_one(HoverTool)

hover = HoverTool(
    tooltips=[
        ("Date",    "@date{%Y-%m-%d}"),
        ("Usage",   "@usage{0.0} MW"),
        ("Rolling", "@rolling{0.0} MW"),
    ],
    formatters={
        '@date': 'datetime'    # key must exactly match the field name
    },
    mode='vline'
)

# 4) Add it
p1.add_tools(hover)

p1.line('date','usage',  source=src1, legend_label='Daily Usage')
p1.line('date','rolling', source=src1, line_color='orange', legend_label='30-Day Rolling Mean')
p1.xaxis.axis_label = 'Date'
p1.yaxis.axis_label = 'Total Daily Usage (MW)'
p1.legend.location = 'top_left'

show(p1)

# —————————————————————————————
# Plot 2: Seasonal decomposition
# —————————————————————————————
decomp   = seasonal_decompose(usage, model='additive', period=365)
trend    = decomp.trend.dropna()
seasonal = decomp.seasonal.loc[trend.index]
resid    = decomp.resid.dropna().loc[trend.index]

# src2 = ColumnDataSource(dict(
#     date=dates,
#     trend=trend,
#     seasonal=seasonal,
#     resid=resid
# ))

# p2 = figure(
#     x_axis_type='datetime', width=700, height=300,
#     title='Seasonal Decomposition of Daily Usage',
#     tools='pan,wheel_zoom,box_zoom,reset,save'
# )

# # 2) Draw each line with its own color
# p2.line('date', 'trend',    source=src2,
#         legend_label='Trend',    line_color='navy',    line_width=2)
# p2.line('date', 'seasonal', source=src2,
#         legend_label='Seasonal', line_color='olive',   line_width=2)
# p2.line('date', 'resid',    source=src2,
#         legend_label='Residual', line_color='firebrick', line_width=2)

# # 3) Axes and legend
# p2.xaxis.axis_label = 'Date'
# p2.yaxis.axis_label = 'Value'
# p2.legend.location = 'top_left'
# p2.title.text_font_size = '14pt'

# # 4) Custom HoverTool
# hover = HoverTool(
#     tooltips=[
#         ("Date",    "@date{%Y-%m-%d}"),
#         ("Value",   "$y{0.0}")
#     ],
#     formatters={'@date': 'datetime'},
#     mode='vline'
# )
# p2.add_tools(hover)

# show(p2)


In [ ]:
## BOX PLOT PER MONTH USAGE

from bokeh.io import output_notebook, output_file, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
import pandas as pd
import numpy as np

output_notebook()
output_file("box plot monthly usage.html", title="histogram usage")

# prepare month‐wise stats
df = df_all.copy()
df['Date'] = pd.to_datetime(df['Date'])
df['month'] = df['Date'].dt.month_name()
months = ['January','February','March','April','May','June',
          'July','August','September','October','November','December']

q1s = []; q2s = []; q3s = []; upp = []; low = []
for m in months:
    vals = df.loc[df['month']==m, 'Total Daily Usage']
    q1 = np.percentile(vals, 25)
    q2 = np.percentile(vals, 50)
    q3 = np.percentile(vals, 75)
    iqr = q3 - q1
    upp.append(min(q3 + 1.5*iqr, vals.max()))
    low.append(max(q1 - 1.5*iqr, vals.min()))
    q1s.append(q1); q2s.append(q2); q3s.append(q3)

source = ColumnDataSource(data=dict(
    month=months, q1=q1s, q2=q2s, q3=q3s, upper=upp, lower=low
))

p = figure(x_range=months, width=800, height=400,
           title='Monthly Distribution of Daily Usage',
           tools='pan,wheel_zoom,box_zoom,reset,save')
# whiskers
p.segment('month','upper', 'month','q3', source=source)
p.segment('month','lower', 'month','q1', source=source)
# boxes
p.vbar('month', 0.7, 'q2','q3', source=source)
p.vbar('month', 0.7, 'q1','q2', source=source)
# caps
p.rect('month','lower', 0.2, 0.01, source=source)
p.rect('month','upper', 0.2, 0.01, source=source)

p.xaxis.major_label_orientation = np.pi/4
p.xaxis.axis_label = 'Month'
p.yaxis.axis_label = 'Total Daily Usage (MW)'
show(p)


In [ ]:
## HISTOGRAM USAGE ANALYSIS PLOT

from bokeh.io import output_notebook, output_file, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
import numpy as np
from scipy.stats import gaussian_kde

usage = df_all['Total Daily Usage']
# overall histogram
hist, edges = np.histogram(usage, bins=30, density=True)
p_hist = figure(width=700, height=400,
                title='Histogram & Density of Daily Usage',
                tools='pan,wheel_zoom,box_zoom,reset,save')
p_hist.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.6)
# KDE
xgrid = np.linspace(usage.min(), usage.max(), 200)
kde = gaussian_kde(usage)
p_hist.line(xgrid, kde(xgrid), line_width=2)
p_hist.xaxis.axis_label = 'Total Daily Usage (MW)'
p_hist.yaxis.axis_label = 'Density'
show(p_hist)

# overlay hot vs cold
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from scipy.stats import gaussian_kde
import numpy as np

usage = df_all['Total Daily Usage']

# determine “hot” vs “cold” days
threshold = df_all['mean_temp'].median()
hot  = df_all.loc[df_all['mean_temp'] >  threshold, 'Total Daily Usage']
cold = df_all.loc[df_all['mean_temp'] <= threshold, 'Total Daily Usage']

# prepare KDEs
xgrid    = np.linspace(usage.min(), usage.max(), 200)
kde_hot  = gaussian_kde(hot)
kde_cold = gaussian_kde(cold)

# overlay plot
p_ov = figure(
    width=700, height=400,
    title='Density: Hot vs Cold Days',
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

# hot days in firebrick (red)
p_ov.line(
    xgrid, kde_hot(xgrid),
    line_width=2,
    line_color='firebrick',
    legend_label='Hot (> median temp)'
)

# cold days in navy (blue)
p_ov.line(
    xgrid, kde_cold(xgrid),
    line_width=2,
    line_color='navy',
    legend_label='Cold (≤ median temp)'
)

p_ov.legend.location = 'top_right'
p_ov.xaxis.axis_label = 'Total Daily Usage (MW)'
p_ov.yaxis.axis_label = 'Density'

show(p_ov)


In [ ]:
from bokeh.io import output_notebook, output_file, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool
import pandas as pd

output_file("daily temp.html", title="daily temp")

# 1) Aggregate mean min/max temp per day
temp_agg = (
    df_all
    .groupby('Date')
    .agg({'mintemp':'mean', 'maxtemp':'mean'})
    .reset_index()
)
temp_agg['Date'] = pd.to_datetime(temp_agg['Date'])

# 2) Create your data source
source = ColumnDataSource(temp_agg)

# 3) Build the figure
p = figure(
    x_axis_type='datetime', width=1000, height=400,
    title="Daily Mean Min & Max Temperature",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)
p.line('Date', 'mintemp', source=source,
       line_color='navy',    line_width=2,
       legend_label="Mean Min Temp")
p.line('Date', 'maxtemp', source=source,
       line_color='firebrick', line_width=2,
       legend_label="Mean Max Temp")

p.xaxis.axis_label = "Date"
p.yaxis.axis_label = "Temperature (°C)"
p.legend.location = "top_left"

# 4) Add a hover that formats Date as YYYY-MM-DD
hover = HoverTool(
    tooltips=[
        ("Date",    "@Date{%Y-%m-%d}"),
        ("Min Temp","@mintemp{0.0} °C"),
        ("Max Temp","@maxtemp{0.0} °C"),
    ],
    formatters={'@Date': 'datetime'},
    mode='vline'
)
p.add_tools(hover)

# 5) Show it
show(p)


In [ ]:
## MODEL PIPELINE

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
from IPython.display import display

df = df_all.copy()

raw_feats = [
    'mintemp', 'maxtemp', 'Total Daily Usage', 'rainfall', 'solar',
    'T_avg', 'HDD_18', 'CDD_18',
    'HDD_3d', 'CDD_3d', 'HDD_7d', 'CDD_7d', 'HDD_14d', 'CDD_14d',
    'heatwave_run', 'coldspell_run'
]

X_raw = df[raw_feats + ['suburb']].copy()
X_train_raw, X_test_raw = train_test_split(
    X_raw, test_size=0.2, random_state=42,
)

hot_thr   = X_train_raw['maxtemp'].quantile(0.90)
cold_thr  = X_train_raw['mintemp'].quantile(0.10)
usage_thr = X_train_raw['Total Daily Usage'].quantile(0.25)


def label(df_):
    extreme = ((df_['maxtemp'] > hot_thr) | (df_['mintemp'] < cold_thr)).astype(int)
    low_use = (df_['Total Daily Usage'] < usage_thr).astype(int)
    return (extreme & low_use).astype(int)

y_train = label(X_train_raw)
y_test  = label(X_test_raw)

X_train = X_train_raw.drop(columns=['mintemp', 'maxtemp', 'Total Daily Usage'])
X_test  = X_test_raw.drop(columns=['mintemp', 'maxtemp', 'Total Daily Usage'])

X_train['suburb_code'], uniques = pd.factorize(X_train.pop('suburb'))
X_test['suburb_code'] = pd.Categorical(X_test.pop('suburb'), categories=uniques).codes

# LightGBM
lgb_train = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test)
params_lgb = {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05, 'verbose': -1, 'seed': 42}
bst_lgb = lgb.train(
    params_lgb, lgb_train, num_boost_round=500,
    valid_sets=[lgb_train, valid_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(20)]
)
pred_lgb = bst_lgb.predict(X_test)
imp_lgb = pd.DataFrame({'feature': bst_lgb.feature_name(), 'importance': bst_lgb.feature_importance('gain')})

# XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
params_xgb = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'eta': 0.05, 'seed': 42}
bst_xgb = xgb.train(
    params_xgb, dtrain, num_boost_round=500,
    evals=[(dtrain, 'train'), (dtest, 'valid')], early_stopping_rounds=50, verbose_eval=20
)
pred_xgb = bst_xgb.predict(dtest)
imp_xgb = pd.DataFrame(bst_xgb.get_score(importance_type='gain').items(), columns=['feature', 'importance'])

# RandomForest
rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict_proba(X_test)[:, 1]
imp_rf = pd.DataFrame({'feature': X_train.columns, 'importance': rf.feature_importances_})

def eval_model(preds):
    auc = roc_auc_score(y_test, preds)
    prec, rec, thresh = precision_recall_curve(y_test, preds)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
    best_thresh = thresh[np.nanargmax(f1_scores)]
    bin_preds = (preds >= best_thresh).astype(int)
    cr = classification_report(y_test, bin_preds, output_dict=True, zero_division=0)
    return {'Threshold': best_thresh, 'AUC': auc, 'Accuracy': cr['accuracy'], 'Precision': cr['1']['precision'], 'Recall': cr['1']['recall'], 'F1': cr['1']['f1-score']}

models = {'LightGBM': pred_lgb, 'XGBoost': pred_xgb, 'RandomForest': pred_rf}
evaluation_df = pd.DataFrame({name: eval_model(pred) for name, pred in models.items()})

feat_imp_df = pd.concat([
    imp_lgb.set_index('feature').rename(columns={'importance': 'LightGBM'}),
    imp_xgb.set_index('feature').rename(columns={'importance': 'XGBoost'}),
    imp_rf.set_index('feature').rename(columns={'importance': 'RandomForest'})
], axis=1).fillna(0)

def compute_risk_series(preds):
    return df.loc[y_test.index].assign(risk=preds).groupby('suburb')['risk'].mean()

risk_df = pd.concat({name: compute_risk_series(pred) for name, pred in models.items()}, axis=1)

display(feat_imp_df)

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 100))
scaled = scaler.fit_transform(feat_imp_df.values)

feat_imp_df = (
    pd.DataFrame(
        scaled,
        index=feat_imp_df.index,
        columns=feat_imp_df.columns
    )
)

display(evaluation_df)
display(feat_imp_df)
display(risk_df)

In [ ]:
## Lgas Data Setup

import pandas as pd
import numpy as np
import geopandas as gpd

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import (
    GeoJSONDataSource, ColumnDataSource, HoverTool,
    LinearColorMapper, ColorBar, BasicTicker
)
from bokeh.tile_providers import CARTODBPOSITRON
from bokeh.palettes import Viridis256

lgas = gpd.read_file('NSW DATA/nsw_localities.shp')

# extract the LOC_NAME column as a list
loc_names = lgas['LOC_NAME'].tolist()

# now loc_names is a Python list
# print(loc_names)

xmin, xmax, ymin, ymax = 151, 155, -34, -33.7
city_slice = lgas.cx[xmin:xmax, ymin:ymax]

target_suburbs = [
    'Epping', 'Macquarie Park', 'Bankstown', 'Concord',
    'Strathfield South', 'Burwood', 'Lidcombe', 'Campsie',
    'Marrickville', 'Leichhardt', 'Crows Nest', 'Surry Hills',
    'Sydney', 'Camperdown', 'Mascot', 'Rockdale',
    'North Sydney', 'Auburn', 'Hurstville', 'Revesby',
    'Terrey Hills', 'Mayfield West', 'Newcastle'
]

extra = lgas[lgas['LOC_NAME'].isin(target_suburbs)]
subset = pd.concat([city_slice, extra], ignore_index=True)
subset = subset.drop_duplicates(subset='LOC_NAME')
subset = gpd.GeoDataFrame(subset, geometry='geometry', crs=lgas.crs)

metrics = pd.DataFrame({
    'LOC_NAME':     target_suburbs,
    'energy_burden': np.random.uniform(5, 25, size=len(target_suburbs)),
    'median_income': np.random.uniform(40000, 90000, size=len(target_suburbs)),
    'population':    np.random.randint(1000, 10000, size=len(target_suburbs)),
    'region_type':   np.random.choice(['Metro','Rural'], size=len(target_suburbs))
})

for col in ['energy_burden','median_income','population','region_type']:
    if col in subset.columns:
        subset = subset.drop(columns=col)
subset = subset.merge(metrics, on='LOC_NAME', how='left')

dt_cols = subset.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns
for col in dt_cols:
    subset[col] = subset[col].dt.strftime("%Y-%m-%dT%H:%M:%S")

subset = subset.to_crs(epsg=3857)

geo_source = GeoJSONDataSource(geojson=subset.to_json())

output_notebook()

In [ ]:
## SUBURB RISK PREDICTION PLOT

import pandas as pd
import geopandas as gpd

from bokeh.io import output_notebook, output_file, show
from bokeh.plotting import figure
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker, HoverTool, GeoJSONDataSource
from bokeh.tile_providers import CARTODBPOSITRON
from bokeh.palettes import Viridis256

# ——————————————————————————————————————
# 1) LOAD & MERGE
# ——————————————————————————————————————

# only name the axis if it's not already named
if risk_df.index.name != 'LOC_NAME':
    risk_df.index.name = 'LOC_NAME'

# only reset if LOC_NAME isn't already a column
if 'LOC_NAME' not in risk_df.columns:
    risk_df = risk_df.reset_index()
else:
    # you already have that column, so just make sure you have a fresh integer index
    risk_df = risk_df.reset_index(drop=True)

# filter your gdf to only the target suburbs
subset = gpd.read_file('NSW DATA/nsw_localities.shp')
gdf = subset[subset['LOC_NAME'].isin(risk_df['LOC_NAME'])].copy()
gdf = gdf.merge(risk_df, on='LOC_NAME', how='left')

# convert to percentages
for m in ['LightGBM','XGBoost','RandomForest']:
    gdf[f"{m}_pct"] = gdf[m] * 100

# — after your merge & pct-conversion, before GeoJSONDataSource ——

# 1) drop any datetime cols so pandas.to_json() won’t choke
datetime_cols = gdf.select_dtypes(include=['datetime64']).columns
if len(datetime_cols):
    gdf = gdf.drop(columns=datetime_cols)

# 2) (optional but recommended) reproject to Web Mercator for the CARTODBPOSITRON tiles
gdf = gdf.to_crs(epsg=3857)

# now serialize exactly as before
geo_source = GeoJSONDataSource(geojson=gdf.to_json())


# ——————————————————————————————————————
# 2) PLOT CHOROPLETH + MULTI‐FIELD HOVER
# ——————————————————————————————————————

output_notebook()

# choose one of the three to map (e.g. LightGBM_pct)
field = "LightGBM_pct"

p = figure(
    title="Risk Prediction (%) by Suburb",
    match_aspect=True,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    x_axis_type="mercator", y_axis_type="mercator",
    width=800, height=600
)
p.add_tile(CARTODBPOSITRON)

# color mapper based on your chosen field
color_mapper = LinearColorMapper(
    palette=Viridis256,
    low=gdf[field].min(),
    high=gdf[field].max()
)

# draw patches
p.patches(
    'xs','ys', source=geo_source,
    fill_color={'field': field,'transform': color_mapper},
    line_color='white', line_width=0.5
)

# add a color bar
color_bar = ColorBar(
    color_mapper=color_mapper,
    ticker=BasicTicker(desired_num_ticks=6),
    label_standoff=8,
    title='Risk (%)',
    location=(0,0)
)
p.add_layout(color_bar, 'right')

# hover tool with all three model percentages
hover = HoverTool(tooltips=[
    ("Suburb",    "@LOC_NAME"),
    ("LightGBM",  "@LightGBM_pct{0.0}%"),
    ("XGBoost",   "@XGBoost_pct{0.0}%"),
    ("RandomForest","@RandomForest_pct{0.0}%")
])
p.add_tools(hover)

show(p)


In [ ]:
## FEATURES IMPORTANCE PLOT

import pandas as pd
from math import pi

from bokeh.io import output_file, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import dodge
from bokeh.palettes import Category10  # vibrant colors

# 1) Prepare data (drop solar, suburb_code, rainfall)
df = (
    feat_imp_df
    .reset_index()
    .rename(columns={'index':'feature'})
    .query("feature not in ['solar','suburb_code','rainfall', 'HDD_3d', 'CDD_3d', 'HDD_7d', 'CDD_7d']")
)

models = ['LightGBM','XGBoost','RandomForest']
source = ColumnDataSource(df)

# 2) Create the figure
output_file("feature_importances_vibrant.html", title="Feature Importances")

p = figure(
    x_range=df['feature'],
    height=450,
    width=700,
    title="Feature Importances by Model",
    toolbar_location="above",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# 3) Draw vibrant bars
offsets = [-0.25, 0.0, 0.25]
palette = Category10[3]

for model, off, color in zip(models, offsets, palette):
    p.vbar(
        x=dodge('feature', off, range=p.x_range),
        top=model,
        width=0.2,
        source=source,
        fill_color=color,
        line_color="white",
        legend_label=model,
        name=model
    )
    
# 4) Add hover tool
hover = HoverTool(
    tooltips=[
        ("Feature", "@feature"),
        ("Model", "$name"),
        ("Importance", "@$name{0.00}")
    ],
    mode="vline"
)
p.add_tools(hover)

# 5) Styling tweaks
p.title.text_font_size = "16pt"
p.title.align = "center"

p.background_fill_color = "#f5f5f5"
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = "white"
p.ygrid.grid_line_width = 1.0

p.xaxis.axis_label = "Features"
p.yaxis.axis_label = "Importance"
p.xaxis.major_label_orientation = pi/4
p.xaxis.major_label_text_font_size = "10pt"
p.yaxis.major_label_text_font_size = "10pt"

p.legend.location = "top_left"
p.legend.background_fill_alpha = 0.0
p.legend.border_line_color = None
p.legend.label_text_font_size = "10pt"

# 6) Show it
show(p)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report

# --- assume df_all, raw_feats, label(), train/test split, and encoding are already done ---
# X_train, X_test, X_train_raw, X_test_raw, y_train, y_test are available from previous steps

# --- Impute missing values via linear interpolation ---
X_train = X_train.interpolate(method='linear', axis=0).fillna(method='bfill').fillna(method='ffill')
X_test  = X_test.interpolate(method='linear', axis=0).fillna(method='bfill').fillna(method='ffill')

# Define simpler models
models = {
    'LogisticRegression': LogisticRegression(solver='liblinear', random_state=42),
    'DecisionTree': DecisionTreeClassifier(max_depth=4, random_state=42)
}

# Containers for predictions and feature importances
y_preds = {}
feat_imps = {}

# Train, predict, and extract feature importance
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    y_preds[name] = preds

    # Coefficients for LR, importances for DT
    if hasattr(model, 'coef_'):
        imp = np.abs(model.coef_[0])
    else:
        imp = model.feature_importances_
    feat_imps[name] = pd.Series(imp, index=X_train.columns)

# Evaluation function
def eval_model(preds):
    auc = roc_auc_score(y_test, preds)
    prec, rec, thresh = precision_recall_curve(y_test, preds)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    best_thresh = thresh[np.nanargmax(f1)]
    bin_preds = (preds >= best_thresh).astype(int)
    cr = classification_report(y_test, bin_preds, output_dict=True, zero_division=0)
    return {
        'Threshold': best_thresh,
        'AUC': auc,
        'Accuracy': cr['accuracy'],
        'Precision': cr['1']['precision'],
        'Recall': cr['1']['recall'],
        'F1': cr['1']['f1-score']
    }

# Compute evaluation DataFrame
evaluation_df = pd.DataFrame(
    {name: eval_model(pred) for name, pred in y_preds.items()}
).T

# Build and scale feature-importance DataFrame
feat_imp_df = pd.DataFrame(feat_imps).fillna(0)
scaler = MinMaxScaler(feature_range=(0, 100))
feat_imp_df.iloc[:, :] = scaler.fit_transform(feat_imp_df)

# Compute suburb-level risk scores using original X_test_raw
risk_series = {}
for name, preds in y_preds.items():
    tmp = pd.DataFrame({
        'suburb': X_test_raw['suburb'],
        'risk': preds
    }, index=X_test_raw.index)
    risk_series[name] = tmp.groupby('suburb')['risk'].mean()
risk_df = pd.concat(risk_series, axis=1)

# Store outputs in dfs
display(evaluation_df)
display(feat_imp_df)
display(risk_df)
